# Project 1: emotion classification on ISEAR

Comparing 3 representations × 3 architectures with cross-validation and a Bayesian ROPE analysis, reusing the tutorial code in `part1/`.

| Section | Who |
| --- | --- |
| 1. Data, 2. Folds, 3. Tokeniser | Britt |
| 4. Representations | Zeno |
| 5. Architectures | Armani |
| 6. Experiments, 7. Bayesian comparison, 8. Error analysis | Together |

Run `python explore_data.py` once first: it creates `data/isear_clean.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import part1
from part1.clslib import NamedClasses

results_folder = Path("results/project1")
results_folder.mkdir(parents=True, exist_ok=True)

## 1. Data
Cleaned ISEAR (see `explore_data.py`): 7,516 raw texts, 84 removed (10 placeholders, 21 same text with conflicting labels, 53 duplicates), 7,432 left. The classes are balanced, so accuracy is a fair metric (majority baseline 14.5%).

In [2]:
LABELS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
outcome_space = NamedClasses(LABELS)

df = pd.read_csv("data/isear_clean.csv")
pairs = list(zip(df["text"], df["label_id"]))  # (text, label) pairs, as in the tutorial
print(len(pairs), "texts")
df["label"].value_counts()

7432 texts


label
joy        1079
anger      1077
fear       1070
disgust    1060
shame      1053
sadness    1050
guilt      1043
Name: count, dtype: int64

## 2. Cross-validation folds
5 folds, repeated with 3 seeds (15 accuracies per system), using the tutorial's `cross_validation_split`. Every system uses these same folds, so the scores are paired for the Bayesian comparison.

In [3]:
seeds = [0, 3, 5]
num_folds = 5
cv_sets = [part1.experiment.cross_validation_split(pairs, k=num_folds, seed=seed) for seed in seeds]

# check: are the classes spread evenly over the folds?
for fold, split in cv_sets[0].items():
    counts = pd.Series([label for _, label in split["validation"]]).value_counts().sort_index()
    print(f"fold {fold}: {len(split['validation'])} texts, per class {counts.tolist()}")

fold 0: 1487 texts, per class [231, 206, 186, 239, 219, 210, 196]
fold 1: 1487 texts, per class [198, 217, 227, 199, 215, 208, 223]
fold 2: 1486 texts, per class [237, 186, 228, 202, 221, 219, 193]
fold 3: 1486 texts, per class [207, 221, 219, 207, 206, 216, 210]
fold 4: 1486 texts, per class [206, 240, 217, 203, 199, 200, 221]


## 3. Tokeniser
BPE tokeniser trained with the tutorial's `train_tokenizer` on the ISEAR texts themselves (~165k words is enough for a small vocabulary). It uses no labels, so training on all texts does not leak label information (the tutorial does the same).

To choose the vocabulary size, we compare how finely each size splits words: a good size keeps frequent words, such as emotion words, whole without making the vocabulary huge.

In [4]:
texts = [text for text, _ in pairs]
n_words = sum(len(t.split()) for t in texts)
example = "I felt ashamed when I shouted at my girlfriend."

for size in [500, 1000, 2000, 5000]:
    tok = part1.experiment.train_tokenizer(texts, size, min_frequency=2)
    n_tokens = sum(len(tok.encode(t).ids) for t in texts)
    print(f"{size:>5}: {n_tokens / n_words:.2f} tokens per word | {tok.encode(example).tokens}")




  500: 2.10 tokens per word | ['▁I', '▁felt', '▁as', 'ham', 'ed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girl', 'friend', '.']



 1000: 1.74 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']



 2000: 1.50 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend.']



 5000: 1.28 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁shouted', '▁at', '▁my', '▁girlfriend.']


In [6]:
vocab_size = 1000  # same as tutorial (500 is too small and with 2000 puctuation gets glued on the words)
tokenizer = part1.experiment.train_tokenizer(texts, vocab_size, min_frequency=2)
tokenizer.save(f"{results_folder}/bpe_vs{vocab_size}.json")
print(tokenizer.encode(example).tokens)




['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']


## 4. Representations (Zeno)
- token indicators (tutorial)
- bag-of-words (tutorial exercise)
- one extra of our choice

## 5. Architectures (Armani)
Log-linear and 1-hidden-layer MLP, plus one extra architecture

In [ ]:
# Log Linear

# f(x, y) -> feature function and returns vectors (in R^D)
# Y -> set of possible labels (7)
# D -> training data (x_i, y_i)
def score(w, x):
    ...
    # return dot(w, f(x, y))

def probabilities(w, x, Y): # turn scores into probabilities
    ...
    # s[y] = score(w, x, y) for every y in Y
    # m = max(s) find the largest score
    # Z = sum over y of exp(s[y]-m) -> softmax denominator. shift by m changes nothing mathematically but avoids bugs
    # return exp(s[y]-m)/Z for every y in Y (softmax probability vector that sums to 1)

def train(D, lr, reg_strength, num_epochs):
    ...
    # w = zeros(D) -> init all weights to zero
    # repeat the next part for every epoch in epochs
    # grad = zeros(D) -> full batch GD, can decide to change later tho
    # for every x, and y (label) in D:
    # p = probabilities(w, x) -> model's curr belief abt each label for datapoint x
    # grad += f(x, y)
    # for y in Y (3rd order loop)
    # grad -= p[y] * f(x, y)
    # grad -= reg_strength * w -> L2 penalty
    # w = w + lr * grad -> step, move w uphill on llh # maxxing the llh, but can also min the -llh
    # return w (learned weights)

def predict(w, x):
    ...
    # return argmax over y in Y of score(w, x, y) (most likely label given x)


## 6. Experiments (together)


## 7. Bayesian comparison with a ROPE (together)
Use tutorial 

## 8. Error analysis (together)
per-class F1 and a confusion matrix